# Telecom Customer Churn Analysis

Exploratory analysis and modeling for the Telco Customer Churn CSV. Place the dataset at `data/telco_customer_churn.csv` before running the cells.

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
from sklearn.cluster import KMeans
from scipy.stats import pearsonr
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import PolynomialFeatures
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

In [ ]:
import pandas as pd
from pathlib import Path

DATA_PATH = Path("data/telco_customer_churn.csv")
if not DATA_PATH.is_file():
    raise FileNotFoundError("Place the Telco Customer Churn CSV at data/telco_customer_churn.csv.")
df = pd.read_csv(DATA_PATH)


In [ ]:
df.head(2)

In [ ]:
df.tail()

In [ ]:
df.shape

In [ ]:
df.info()

In [ ]:
df.describe()

In [ ]:
nulls_in_columns = df.isnull().sum()
print("Null values in each column:")
print(nulls_in_columns)

In [ ]:
nulls_in_rows = df.isnull().sum(axis=1)
print("Null values in each row:")
print(nulls_in_rows)

In [ ]:
nulls_in_columns = df.isnull().sum()
print("Null values in each column:")
print(nulls_in_columns)

In [ ]:
df.isnull().sum()

In [ ]:
column_to_encoded = 'Churn'
one_hot_encoded=pd.get_dummies(df[column_to_encoded],prefix=column_to_encoded)
df=pd.concat([df,one_hot_encoded],axis=1)
df

In [ ]:
df["Churn_Yes_Num"]=df["Churn"].map({"Yes":1,"No":0})
df

In [ ]:
for column in df.columns:
  unique_values=df[column].unique()
  print(f'uique value in {column}: ',unique_values)

In [ ]:
df.drop('TotalCharges',axis=1,inplace=True)

In [ ]:
fig,ax=plt.subplots()
sns.histplot(df['tenure'],kde=True)
plt.title('distribution of tenure')

In [ ]:
fig,axs=plt.subplots(1,2, figsize=(17,5))

axs[0].hist(df['Contract'])
axs[0].set_title('contract type distribution')

axs[1].hist(df["PaymentMethod"])
axs[1].set_title('payment method distribution')

In [ ]:
sns.countplot(x='Churn',data=df)
plt.title('churn distribution')

In [ ]:
sum_variable=df["Churn_Yes_Num"].sum()
sum_variable

In [ ]:
count_values=df["Churn_Yes_Num"].count()
count_values
total_no=count_values-sum_variable
total_no

In [ ]:
contract_churn=df.groupby("Contract")['Churn_Yes_Num'].mean()
payment_churn=df.groupby("PaymentMethod")["Churn_Yes_Num"].mean()

plt.figure(figsize=(6,7))
sns.barplot(x=contract_churn.index,y=contract_churn.values)
plt.title("churn rate by contract type")
plt.ylabel('churn rate')
plt.show()


In [ ]:
plt.figure(figsize=(12,12))
sns.barplot(x=payment_churn.index,y=payment_churn.values)
plt.title("churn rate by payment method")
plt.ylabel('churn rate')
plt.show()

**customer segmentation**

In [ ]:
def segment_tenure(tenure):
  if tenure < 1:
    return "new costumer"
  elif tenure>=1 and tenure<=1:
    return "mid term customer"
  else:
    return "long term costumer"

df["tenure_segment"]=df['tenure'].apply(segment_tenure)

def segment_charges(charges):
  if charges < 50:
    return "low charge"
  elif charges>=50 and charges<=100:
    return "mid charge"
  else:
    return "high charge"

df["charges_segment"]=df['MonthlyCharges'].apply(segment_charges)


def segment_contract_type(contract_type):
  if contract_type == 'Month-to-Month':
    return "month to month contract"
  elif contract_type == 'One year':
    return "one year contract"
  else:
    return "two year contract"

df["contract_segment"]=df['Contract'].apply(segment_contract_type)

df[["tenure_segment","charges_segment","contract_segment"]]
df





In [ ]:
def calculate_churn_rate(segment):
  total_customers=len(segment)
  churned_customers=segment['Churn_Yes_Num'].sum()
  churn_rate = (churned_customers / total_customers)*100
  return churn_rate

tenure_segment=df.groupby('tenure_segment')
tenure_churn_rate=tenure_segment.apply(calculate_churn_rate)

In [ ]:
def calculate_churn_rate(segment):
    total_customers = len(segment)
    churned_customers = segment['Churn_Yes'].sum()
    churn_rate = (churned_customers / total_customers) * 100
    return churn_rate

# Segment customers based on tenure
tenure_segment = df.groupby('tenure_segment')
tenure_churn_rate = tenure_segment.apply(calculate_churn_rate)

# Segment customers based on monthly charges
charges_segment = df.groupby('charges_segment')
charges_churn_rate = charges_segment.apply(calculate_churn_rate)

# Segment customers based on contract type
contract_segment = df.groupby('contract_segment')
contract_churn_rate = contract_segment.apply(calculate_churn_rate)

In [ ]:
# Segment customers based on tenure
customer_segments = df.groupby('tenure_segment')
segment_churn_rates = customer_segments.apply(calculate_churn_rate)

In [ ]:
def calculate_churn_rates(df):
    segments = define_segments(df)
    churn_rates = calculate_rates(segments)
    print_rates(churn_rates)

In [ ]:
def calculate_churn_rate(segment):
    total_customers = len(segment)
    churned_customers = segment['Churn_Yes'].sum()
    churn_rate = (churned_customers / total_customers) * 100
    return churn_rate

def calculate_churn_rates(df):
    segments = define_segments(df)
    churn_rates = calculate_rates(segments)
    print_rates(churn_rates)

def define_segments(df):
    return df.groupby(['tenure_segment', 'charges_segment', 'contract_segment'])

def calculate_rates(segments):
    return segments.apply(calculate_churn_rate)

def print_rates(churn_rates):
    print(churn_rates)


In [ ]:
churn_rates = {
    'tenure': segment_churn_rates,
    'charges': charges_churn_rate,
    'contract': contract_churn_rate
}

print(churn_rates)


In [ ]:
import matplotlib.pyplot as plt

# Extracting churn rates for tenure, charges, and contract segments
tenure = churn_rates['tenure'].index.tolist()
tenure_rates = churn_rates['tenure'].values

charges = churn_rates['charges'].index.tolist()
charges_rates = churn_rates['charges'].values

contract = churn_rates['contract'].index.tolist()
contract_rates = churn_rates['contract'].values

# Plotting churn rates by tenure segment
plt.bar(tenure, tenure_rates)
plt.xlabel('Tenure Segment')
plt.ylabel('Churn Rate %')
plt.title('Churn Rate by Tenure Segment')
plt.show()

# Plotting churn rates by charges segment
plt.bar(charges, charges_rates)
plt.xlabel('Charges Segment')
plt.ylabel('Churn Rate %')
plt.title('Churn Rate by Charges Segment')
plt.show()


In [ ]:
import matplotlib.pyplot as plt

# Plotting churn rates by contract segment
plt.bar(contract, contract_rates)
plt.xlabel('Contract Type')
plt.ylabel('Churn Rate %')
plt.title('Churn Rate by Contract Segment')
plt.show()


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, roc_auc_score

# Initialize LabelEncoder
le = LabelEncoder()

# Encode categorical columns
for col in ['customerID', 'gender', 'SeniorCitizen', 'Partner', 'Dependents', 'PhoneService', 'MultipleLines',
            'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV',
            'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod', 'tenure_segment', 'charges_segment',
            'contract_segment', 'Churn']:
    df[col] = le.fit_transform(df[col])

# Split data
X_train, X_test, y_train, y_test = train_test_split(df.drop('Churn_Yes_Num', axis=1), df['Churn_Yes_Num'], test_size=0.25, random_state=42)

# Train logistic regression model
model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)
y_pred = model.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred))
print("AUC-ROC:", roc_auc_score(y_test, y_pred))


In [ ]:
import seaborn as sns
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, roc_auc_score

# Split data
X_train, X_test, y_train, y_test = train_test_split(df[['tenure']], df['Churn_Yes_Num'], test_size=0.25, random_state=42)

# Train logistic regression model
model = LogisticRegression(max_iter=5)
model.fit(X_train, y_train)
y_pred = model.predict(X_test)

print('Accuracy:', accuracy_score(y_test, y_pred))
print('AUC-ROC:', roc_auc_score(y_test, y_pred))

plt.figure(figsize=(10, 6))
sns.scatterplot(x='tenure', y='Churn_Yes_Num', hue='Churn_Yes_Num', data=df, palette='coolwarm', marker='o', s=75)

# Plot the logistic regression line
x_range = np.linspace(X_test['tenure'].min(), X_test['tenure'].max(), 300).reshape(-1, 1)
y_proba = model.predict_proba(x_range)[:, 1]
plt.plot(x_range, y_proba, color='red', label='Logistic Regression Line')

# Add labels and legend
plt.xlabel('Tenure')
plt.ylabel('Probability')
plt.legend()
plt.title('Logistic Regression Line Visualization')
plt.show()


In [ ]:
import numpy as np
from sklearn.metrics import accuracy_score

# Check for NaN values in each column
for col in df.columns:
    if df[col].isnull().any():
        print(f"Column '{col}' has NaN values.")

# Step 1: Make predictions on the test set using the tuned model
y_pred_test = model.predict(X_test)

# Step 2: Calculate the test accuracy
test_accuracy = accuracy_score(y_test, y_pred_test)

# Step 3: Print the test accuracy
print(f"Test Accuracy: {test_accuracy:.2f}")


In [ ]:
def calculate_ltv(dataframe):
    # Filter data for active customers (not churned)
    active_customers = dataframe[dataframe['Churn'] == 0]

    # Calculate the average monthly charges for active customers
    avg_monthly_charges = active_customers['MonthlyCharges'].mean()

    # Calculate the ratio of active customers to the total number of unique customers
    ratio_active_customers = active_customers['customerID'].count() / active_customers['customerID'].nunique()

    # Calculate the retention rate
    retention_rate = 1 / (1 - (dataframe['Churn'].sum() / dataframe['customerID'].nunique()))

    # Calculate the Customer Lifetime Value (LTV)
    ltv = avg_monthly_charges * ratio_active_customers * retention_rate

    return ltv

# Calculate the LTV for the given DataFrame
ltv_result = calculate_ltv(df)

# Print the Customer Lifetime Value (LTV)
print(f"Customer Lifetime Value (LTV): ${ltv_result:,.2f}")


In [ ]:
# Assuming you have trained a logistic regression model named 'model'
coefficients = model.coef_[0]

# Assuming 'X' represents the feature matrix used for training
feature_names = X.columns

# Calculate the indices that would sort the coefficients by their absolute values in descending order
sorted_indices = np.argsort(np.abs(coefficients))[::-1]

# Print feature importance
for i, idx in enumerate(sorted_indices):
    feature_name = feature_names[idx]
    importance = coefficients[idx]
    print(f"{i + 1}. {feature_name}: {importance:.4f}")

# Plot feature importances
plt.figure(figsize=(10, 6))
plt.bar(range(len(coefficients)), coefficients[sorted_indices], align="center")
plt.xticks(range(len(coefficients)), [feature_names[i] for i in sorted_indices], rotation=45)
plt.xlabel("Feature")
plt.ylabel("Importance")
plt.title("Feature Importances")
plt.show()


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report

X = df[['tenure', 'MonthlyCharges', 'Contract', 'TechSupport', 'StreamingTV']]
y = df['Churn']

# Splitting the data into training and testing sets
x_train, x_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Model training
rf_model = RandomForestClassifier()
rf_model.fit(x_train, y_train)

# Making predictions
predictions = rf_model.predict(x_test)

# Print classification report
print(classification_report(y_test, predictions))

# Identify high-value customers at risk of churning
df['Churn_Predicted'] = rf_model.predict(X)
threshold_value = 0.5  # Adjust this threshold based on your business context
high_value_churn_risk_customers = df[(df['Churn_Predicted'] == 1) & (df['MonthlyCharges'] > threshold_value)]


In [ ]:
print("The total high_value_churn_risk_customers are:", high_value_churn_risk_customers['customerID'].count())



In [ ]:
plt.figure(figsize=(10, 6))
plt.plot(df['tenure'], df['Churn'], marker='o', linestyle='-', color='b')
plt.title('Churn Over Time')
plt.xlabel('Tenure')
plt.ylabel('Churn Status (1 for Churned, 0 for Active)')
plt.grid(True)
plt.show()


In [ ]:
plt.figure(figsize=(10, 6))
sns.countplot(x='Contract', hue='Churn', data=df, palette='viridis')
plt.title('Churn Count by Contract Type')
plt.xlabel('Contract Type')
plt.ylabel('Count')
plt.show()


In [ ]:
plt.figure(figsize=(8, 8))
df['Partner'].value_counts().plot.pie(autopct='%1.1f%%', colors=['lightcoral', 'lightskyblue'])
plt.title('Proportion of Customers with Partners')
plt.ylabel('')
plt.show()

In [ ]:
correlation_matrix = df.corr()
plt.figure(figsize=(10, 8))
sns.heatmap(correlation_matrix, cmap='coolwarm', annot=True, fmt=".2f", linewidths=0.5)
plt.title('Correlation Matrix of Numeric Features')
plt.show()


In [ ]:
plt.figure(figsize=(10, 6))
sns.scatterplot(x='tenure', y='MonthlyCharges', hue='Churn', data=df, palette='Set1')
plt.title('Scatter Plot of Tenure vs. Monthly Charges')
plt.xlabel('Tenure')
plt.ylabel('Monthly Charges')
plt.show()


In [ ]:
plt.figure(figsize=(10, 6))
sns.boxplot(x='Contract', y='MonthlyCharges', data=df, palette='Pastel1')
plt.title('Monthly Charges Distribution by Contract Type')
plt.xlabel('Contract Type')
plt.ylabel('Monthly Charges')
plt.show()


In [ ]:
plt.figure(figsize=(10, 6))
sns.violinplot(x='Churn', y='MonthlyCharges', data=df, palette='Set2')
plt.title('Distribution of Monthly Charges by Churn Status')
plt.xlabel('Churn Status')
plt.ylabel('Monthly Charges')
plt.show()


In [ ]:
plt.figure(figsize=(10, 6))
sns.countplot(x='InternetService', hue='Churn', data=df, palette='muted')
plt.title('Churn Count by Internet Service Type')
plt.xlabel('Internet Service Type')
plt.ylabel('Count')
plt.show()


In [ ]:
categories = ['TechSupport', 'OnlineSecurity', 'StreamingTV', 'Churn']

plt.figure(figsize=(6, 6))
for category in categories:
    values = df.groupby(category).size().reset_index(name='Count')
    values['Count'] = values['Count'] / len(df)
    plt.polar(np.linspace(0, 2 * np.pi, len(values)), values['Count'], label=category)

plt.title('Comparison of Satisfaction Scores by Service Category')
plt.legend()
plt.show()
